In [10]:
from pathlib import Path
import numpy as np
import pandas as pd
import re
from tqdm.auto import tqdm

def locate_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists():
            return candidate
    return current

root = locate_root()
data_dir = root / "data"
metadata_dir = data_dir / "metadata"
interim_dir = data_dir / "interim"
processed_dir = data_dir / "processed"
reports_dir = root / "reports"
figures_dir = reports_dir / "figures"
tables_dir = reports_dir / "tables"

for directory in [interim_dir, processed_dir, figures_dir, tables_dir]:
    directory.mkdir(parents=True, exist_ok=True)

profile_df = pd.read_csv(metadata_dir / "dataset_profile.csv")
column_frequency = pd.read_csv(metadata_dir / "column_frequency.csv")
healthy_common_columns = pd.read_csv(metadata_dir / "common_columns_healthy.csv")["column_name"].dropna().astype(str).tolist()

profile_df["header_row"] = pd.to_numeric(profile_df["header_row"], errors="coerce")
profile_df = profile_df[profile_df["error"].isna()].copy()
profile_df = profile_df[(profile_df["rows"].fillna(0) >= 100) & (profile_df["columns"].fillna(0) >= 10)].copy()

healthy_profile = profile_df[profile_df["group"] == "healthy"].reset_index(drop=True)
defective_profile = profile_df[profile_df["group"] == "defective"].reset_index(drop=True)
valid_profile = pd.concat([healthy_profile, defective_profile], ignore_index=True)

pd.DataFrame(
    {
        "metric": ["healthy_files", "defective_files", "valid_files", "healthy_common_columns"],
        "value": [
            healthy_profile["file_name"].nunique(),
            defective_profile["file_name"].nunique(),
            valid_profile["file_name"].nunique(),
            len(healthy_common_columns),
        ],
    }
)

,metric,value
0,healthy_files,9
1,defective_files,14
2,valid_files,23
3,healthy_common_columns,196


In [11]:
def normalize_columns(columns):
    normalized = []
    for value in columns:
        text = "" if pd.isna(value) else str(value).strip().lower()
        text = re.sub(r"\s+", "_", text)
        text = re.sub(r"[^a-z0-9_]", "", text)
        normalized.append(text if text else "unnamed")
    seen = {}
    output = []
    for name in normalized:
        seen[name] = seen.get(name, 0) + 1
        output.append(name if seen[name] == 1 else f"{name}_{seen[name]}")
    return output

def choose_header_from_preview(preview):
    preview = preview.dropna(axis=1, how="all").dropna(axis=0, how="all")
    if preview.empty:
        return None
    best_header = None
    best_score = -1
    ncols = preview.shape[1]
    for h in [0, 1, 2, 3]:
        if h >= len(preview):
            continue
        cols = normalize_columns(preview.iloc[h].tolist())
        score = sum(not str(col).startswith("unnamed") for col in cols)
        if score > best_score:
            best_score = score
            best_header = h
    if best_header is None:
        return None
    if best_score < max(1, int(ncols * 0.4)):
        return None
    return best_header

def read_frame(path, sheet_name, header_row):
    header = None if pd.isna(header_row) else int(header_row)
    with pd.ExcelFile(path, engine="openpyxl") as xls:
        if header is None:
            preview = pd.read_excel(xls, sheet_name=sheet_name, header=None, nrows=6)
            header = choose_header_from_preview(preview)
        df = pd.read_excel(xls, sheet_name=sheet_name, header=header)
    df = df.dropna(axis=1, how="all").dropna(axis=0, how="all")
    df.columns = normalize_columns(df.columns)
    return df

def detect_time_source(df):
    pattern = re.compile(r"(^|_)(time|timestamp|date|sec|second|minute|hour|ms)(_|$)")
    candidates = [col for col in df.columns if pattern.search(str(col))]
    best_col = None
    best_score = -1.0
    for col in candidates:
        s = pd.to_numeric(df[col], errors="coerce")
        valid = s.dropna()
        if len(valid) < max(20, int(len(df) * 0.1)):
            continue
        monotonic_score = float((valid.diff().fillna(0) >= 0).mean())
        uniqueness_score = float(valid.nunique() / max(len(valid), 1))
        span_score = float(valid.max() > valid.min())
        score = monotonic_score * 2.0 + uniqueness_score + span_score
        if score > best_score:
            best_score = score
            best_col = col
    return best_col

def build_time_index(df):
    time_source = detect_time_source(df)
    if time_source is None:
        values = np.arange(len(df), dtype=np.float32)
        return pd.Series(values, index=df.index), "sequence_index"
    s = pd.to_numeric(df[time_source], errors="coerce")
    if s.notna().sum() == 0:
        values = np.arange(len(df), dtype=np.float32)
        return pd.Series(values, index=df.index), "sequence_index"
    s = s.interpolate(limit_direction="both")
    if s.notna().sum() == 0:
        values = np.arange(len(df), dtype=np.float32)
        return pd.Series(values, index=df.index), "sequence_index"
    base = float(s.dropna().iloc[0])
    values = (s - base).astype(np.float32)
    if not np.isfinite(values).all():
        values = pd.Series(np.arange(len(df), dtype=np.float32), index=df.index)
        return values, "sequence_index"
    return values, time_source

def prepare_numeric_block(df, feature_candidates):
    work = df.reindex(columns=feature_candidates).copy()
    for col in work.columns:
        work[col] = pd.to_numeric(work[col], errors="coerce")
    return work.astype(np.float32)

In [12]:
defective_file_count = defective_profile["file_name"].nunique()
min_defective_coverage = max(1, int(np.ceil(defective_file_count * 0.6)))
exclude_pattern = re.compile(r"^(unnamed.*|index|row_index|sequence_index)$|(^|_)(time|timestamp|date)(_|$)")

feature_candidates = column_frequency[
    (column_frequency["column_name"].isin(healthy_common_columns)) &
    (column_frequency["defective_file_count"] >= min_defective_coverage)
]["column_name"].dropna().astype(str).tolist()

feature_candidates = [col for col in feature_candidates if not exclude_pattern.search(col)]

if len(feature_candidates) == 0:
    raise ValueError("No feature candidates were found for preprocessing.")

pd.DataFrame(
    {
        "metric": ["feature_candidates", "min_defective_coverage"],
        "value": [len(feature_candidates), min_defective_coverage],
    }
)

,metric,value
0,feature_candidates,196
1,min_defective_coverage,9


In [13]:
prepared_parts = []
time_manifest_rows = []

for row in tqdm(valid_profile.itertuples(index=False), total=len(valid_profile), desc="Preparing flights"):
    path = root / row.relative_path
    df = read_frame(path, row.sheet_name, row.header_row)
    time_index, time_source = build_time_index(df)
    numeric_block = prepare_numeric_block(df, feature_candidates)
    row_non_null_ratio = numeric_block.notna().mean(axis=1) if numeric_block.shape[1] else pd.Series(np.zeros(len(numeric_block)), index=numeric_block.index)
    keep_rows = row_non_null_ratio >= 0.02
    if keep_rows.sum() == 0:
        continue
    block = numeric_block.loc[keep_rows].reset_index(drop=True)
    original_row_index = pd.Series(np.flatnonzero(keep_rows.to_numpy()), dtype=np.int32)
    kept_time_index = time_index.loc[keep_rows].reset_index(drop=True).astype(np.float32)
    part = pd.DataFrame(
        {
            "file_name": row.file_name,
            "group": row.group,
            "relative_path": row.relative_path,
            "sheet_name": row.sheet_name,
            "header_row": int(row.header_row) if pd.notna(row.header_row) else np.nan,
            "original_row_index": original_row_index,
            "sequence_index": np.arange(len(block), dtype=np.int32),
            "time_index": kept_time_index,
            "time_source": time_source,
        }
    )
    part = pd.concat([part, block], axis=1)
    prepared_parts.append(part)
    time_manifest_rows.append(
        {
            "file_name": row.file_name,
            "group": row.group,
            "sheet_name": row.sheet_name,
            "time_source": time_source,
            "rows_loaded": int(len(block)),
            "feature_columns_present": int(block.notna().any(axis=0).sum()),
        }
    )

prepared_df = pd.concat(prepared_parts, ignore_index=True) if prepared_parts else pd.DataFrame(
    columns=["file_name", "group", "relative_path", "sheet_name", "header_row", "original_row_index", "sequence_index", "time_index", "time_source", *feature_candidates]
)
time_manifest_df = pd.DataFrame(time_manifest_rows)

metadata_columns = ["file_name", "group", "relative_path", "sheet_name", "header_row", "original_row_index", "sequence_index", "time_index", "time_source"]
feature_columns = [col for col in prepared_df.columns if col not in metadata_columns]

pd.DataFrame(
    {
        "metric": ["prepared_rows", "prepared_features", "prepared_files"],
        "value": [len(prepared_df), len(feature_columns), prepared_df["file_name"].nunique()],
    }
)

Preparing flights:   0%|          | 0/23 [00:00<?, ?it/s]

Preparing flights: 100%|██████████| 23/23 [11:24<00:00, 29.76s/it]


,metric,value
0,prepared_rows,312501
1,prepared_features,196
2,prepared_files,23


In [14]:
feature_manifest_rows = []

for col in tqdm(feature_columns, total=len(feature_columns), desc="Building feature manifest"):
    s_all = prepared_df[col]
    s_h = prepared_df.loc[prepared_df["group"] == "healthy", col]
    s_d = prepared_df.loc[prepared_df["group"] == "defective", col]
    feature_manifest_rows.append(
        {
            "feature": col,
            "all_non_null_ratio": float(s_all.notna().mean()),
            "healthy_non_null_ratio": float(s_h.notna().mean()) if len(s_h) else np.nan,
            "defective_non_null_ratio": float(s_d.notna().mean()) if len(s_d) else np.nan,
            "healthy_file_coverage": int(prepared_df.loc[(prepared_df["group"] == "healthy") & prepared_df[col].notna(), "file_name"].nunique()),
            "defective_file_coverage": int(prepared_df.loc[(prepared_df["group"] == "defective") & prepared_df[col].notna(), "file_name"].nunique()),
        }
    )

feature_manifest = pd.DataFrame(feature_manifest_rows).sort_values(
    ["healthy_non_null_ratio", "defective_non_null_ratio", "feature"],
    ascending=[False, False, True],
).reset_index(drop=True)

selected_features = feature_manifest[
    (feature_manifest["healthy_non_null_ratio"].fillna(0) >= 0.5) &
    (feature_manifest["defective_non_null_ratio"].fillna(0) >= 0.3)
]["feature"].tolist()

if len(selected_features) == 0:
    raise ValueError("No selected features remained after coverage filtering.")

prepared_df = prepared_df[metadata_columns + selected_features].copy()
feature_columns = selected_features

pd.DataFrame(
    {
        "metric": ["selected_features", "mean_all_non_null_ratio", "mean_healthy_non_null_ratio", "mean_defective_non_null_ratio"],
        "value": [
            len(feature_columns),
            float(feature_manifest[feature_manifest["feature"].isin(feature_columns)]["all_non_null_ratio"].mean()),
            float(feature_manifest[feature_manifest["feature"].isin(feature_columns)]["healthy_non_null_ratio"].mean()),
            float(feature_manifest[feature_manifest["feature"].isin(feature_columns)]["defective_non_null_ratio"].mean()),
        ],
    }
)

Building feature manifest:   0%|          | 0/196 [00:00<?, ?it/s]

Building feature manifest: 100%|██████████| 196/196 [00:21<00:00,  9.21it/s]


,metric,value
0,selected_features,178.0
1,mean_all_non_null_ratio,1.0
2,mean_healthy_non_null_ratio,1.0
3,mean_defective_non_null_ratio,1.0


In [15]:
healthy_block = prepared_df.loc[prepared_df["group"] == "healthy", feature_columns].copy()
healthy_median = healthy_block.median()
healthy_q25 = healthy_block.quantile(0.25)
healthy_q75 = healthy_block.quantile(0.75)
healthy_iqr = healthy_q75 - healthy_q25
healthy_std = healthy_block.std(ddof=0)
healthy_mean = healthy_block.mean()
healthy_mad = (healthy_block - healthy_median).abs().median()

scale = healthy_iqr.replace(0, np.nan)
scale = scale.fillna((1.4826 * healthy_mad).replace(0, np.nan))
scale = scale.fillna(healthy_std.replace(0, np.nan))
scale = scale.fillna(1.0).astype(np.float32)

standardized_df = prepared_df.copy()
standardized_df[feature_columns] = ((prepared_df[feature_columns] - healthy_median) / scale).astype(np.float32)

healthy_reference_stats = pd.DataFrame(
    {
        "feature": feature_columns,
        "healthy_mean": healthy_mean.reindex(feature_columns).to_numpy(dtype=np.float64),
        "healthy_median": healthy_median.reindex(feature_columns).to_numpy(dtype=np.float64),
        "healthy_std": healthy_std.reindex(feature_columns).to_numpy(dtype=np.float64),
        "healthy_q25": healthy_q25.reindex(feature_columns).to_numpy(dtype=np.float64),
        "healthy_q75": healthy_q75.reindex(feature_columns).to_numpy(dtype=np.float64),
        "healthy_iqr": healthy_iqr.reindex(feature_columns).to_numpy(dtype=np.float64),
        "healthy_mad": healthy_mad.reindex(feature_columns).to_numpy(dtype=np.float64),
        "robust_scale": scale.reindex(feature_columns).to_numpy(dtype=np.float64),
    }
).sort_values("feature").reset_index(drop=True)

file_row_counts = prepared_df.groupby(["group", "file_name"], as_index=False).agg(
    rows=("sequence_index", "count"),
    time_source=("time_source", "first"),
    min_time=("time_index", "min"),
    max_time=("time_index", "max"),
)

file_row_counts

,group,file_name,rows,time_source,min_time,max_time
0,defective,Defect15(01).xlsx,12431,sequence_index,0.0,12430.0
1,defective,Defect15(02).xlsx,4902,sequence_index,0.0,4901.0
2,defective,Defect15(03).xlsx,15689,sequence_index,0.0,15688.0
3,defective,Defect15(04).xlsx,13123,sequence_index,0.0,13122.0
4,defective,Defect16(01).xlsx,9690,sequence_index,0.0,9689.0
5,defective,Defect16(02).xlsx,14537,sequence_index,0.0,14536.0
6,defective,Defect16(03).xlsx,14683,sequence_index,0.0,14682.0
7,defective,Defected 1.xlsx,12431,sequence_index,0.0,12430.0
8,defective,Defected 2.xlsx,4902,sequence_index,0.0,4901.0
9,defective,Defected 3.xlsx,15689,sequence_index,0.0,15688.0


In [16]:
raw_path = interim_dir / "flight_rows_raw.parquet"
standardized_path = processed_dir / "flight_rows_standardized.parquet"
feature_manifest_path = metadata_dir / "feature_manifest.csv"
healthy_stats_path = metadata_dir / "healthy_reference_feature_stats.csv"
time_manifest_path = metadata_dir / "time_axis_manifest.csv"
file_row_counts_path = tables_dir / "prepared_file_row_counts.csv"
preparation_summary_path = tables_dir / "preparation_summary.csv"

prepared_df.to_parquet(raw_path, index=False)
standardized_df.to_parquet(standardized_path, index=False)
feature_manifest.to_csv(feature_manifest_path, index=False)
healthy_reference_stats.to_csv(healthy_stats_path, index=False)
time_manifest_df.to_csv(time_manifest_path, index=False)
file_row_counts.to_csv(file_row_counts_path, index=False)
pd.DataFrame(
    {
        "metric": ["rows", "features", "files", "healthy_rows", "defective_rows"],
        "value": [
            len(prepared_df),
            len(feature_columns),
            prepared_df["file_name"].nunique(),
            int((prepared_df["group"] == "healthy").sum()),
            int((prepared_df["group"] == "defective").sum()),
        ],
    }
).to_csv(preparation_summary_path, index=False)

{
    "flight_rows_raw": str(raw_path),
    "flight_rows_standardized": str(standardized_path),
    "feature_manifest": str(feature_manifest_path),
    "healthy_reference_feature_stats": str(healthy_stats_path),
    "time_axis_manifest": str(time_manifest_path),
    "prepared_file_row_counts": str(file_row_counts_path),
    "preparation_summary": str(preparation_summary_path),
}

{'flight_rows_raw': '/home/hamza/Documents/aero-res/aircraft_fault_localization/data/interim/flight_rows_raw.parquet',
 'flight_rows_standardized': '/home/hamza/Documents/aero-res/aircraft_fault_localization/data/processed/flight_rows_standardized.parquet',
 'feature_manifest': '/home/hamza/Documents/aero-res/aircraft_fault_localization/data/metadata/feature_manifest.csv',
 'healthy_reference_feature_stats': '/home/hamza/Documents/aero-res/aircraft_fault_localization/data/metadata/healthy_reference_feature_stats.csv',
 'time_axis_manifest': '/home/hamza/Documents/aero-res/aircraft_fault_localization/data/metadata/time_axis_manifest.csv',
 'prepared_file_row_counts': '/home/hamza/Documents/aero-res/aircraft_fault_localization/reports/tables/prepared_file_row_counts.csv',
 'preparation_summary': '/home/hamza/Documents/aero-res/aircraft_fault_localization/reports/tables/preparation_summary.csv'}